In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
from pinecone import Pinecone, ServerlessSpec

In [3]:
# Pinecone 연결
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "quickstart-index"

# 인덱스가 없으면 새로 생성
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 인덱스 연결
index = pc.Index(index_name)

In [4]:
from openai import OpenAI
import time

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 저장할 문서
documents = [
    {"id": "doc1", "text": "Pinecone은 벡터 데이터베이스입니다."},
    {"id": "doc2", "text": "Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다."},
]

# 문장 → 임베딩 벡터
response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[document["text"] for document in documents],
)

vectors = [
    {
        "id": document["id"],
        "values": embedding.embedding,
        "metadata": {
            "text": document["text"],
            "category": "tech"
        },
    }
    for document, embedding in zip(documents, response.data)
]

# Pinecone에 저장
index.upsert(
    vectors=vectors,
    namespace="example-namespace"
)

time.sleep(2)

In [5]:
# 검색할 질문
question = "임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?"

# 질문도 임베딩 벡터로 변환
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

# 질문과 의미가 가까운 문서 검색
result = index.query(
    vector=question_vector,
    top_k=2,
    include_metadata=True,
    namespace="example-namespace"
)

print(f"질문: {question}\n")

for match in result.matches:
    print(
        f"{match.id}: {match.metadata['text']} "
        f"(유사도: {match.score:.4f})"
    )

질문: 임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?

doc2: Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다. (유사도: 0.6642)
doc1: Pinecone은 벡터 데이터베이스입니다. (유사도: 0.4800)
